In [ ]:
# ============================================================
# STEP 3 - DEEP GRAPH INFOMAX (DGI) EMBEDDINGS
# Original outputs + additional training-time embedding experiment
# ============================================================

# ============================================================
# CELL 1 - Imports
# ============================================================

import os
import pickle
import random
import numpy as np
import pandas as pd
import torch
import networkx as nx

from tqdm import tqdm

from torch_geometric.nn import GCNConv
from torch_geometric.nn.models import DeepGraphInfomax
from torch_geometric.utils import from_networkx
from torch_geometric.loader import DataLoader



c:\Users\Giuliana Galassi\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# CELL 2 - Reproducibility and device
# ============================================================

def set_seed(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# ============================================================
# CELL 3 - Paths
# ============================================================

graphs_path = "facebook0_link_prediction/graphs_adamic_adar_nsum.pkl"
features_path = "features_aligned_facebook0.csv"
ard_path = "ard_matrix_facebook0.csv"
original_edges_path = "0.edges"

output_folder = "facebook0_dgi"
os.makedirs(output_folder, exist_ok=True)

In [4]:
# ============================================================
# CELL 4 - Load synthetic graphs
# ============================================================

def load_augmented_graphs(graphs_path):
    with open(graphs_path, "rb") as f:
        graphs = pickle.load(f)
    return graphs


graphs = load_augmented_graphs(graphs_path)

print("Number of synthetic graphs:", len(graphs))
print("Example synthetic graph nodes:", graphs[0].number_of_nodes())
print("Example synthetic graph edges:", graphs[0].number_of_edges())

Number of synthetic graphs: 450
Example synthetic graph nodes: 333
Example synthetic graph edges: 2489


In [5]:
# ============================================================
# CELL 5 - Load all features, ARD circles-only features,
#          and graph-structure-only identity features
# ============================================================

def load_features_all(features_path):
    features_df = pd.read_csv(features_path, index_col=0)

    features_df = features_df.reset_index(drop=True)
    features_df.index = range(1, len(features_df) + 1)
    features_df.index.name = "node_position"

    features_tensor = torch.tensor(
        features_df.to_numpy(),
        dtype=torch.float32
    )

    return features_df, features_tensor


def load_ard_features(ard_path):
    """
    Loads the already processed ARD matrix.

    This matrix should already be built from:
    - significant circles
    - circleNA
    """

    ard_df = pd.read_csv(ard_path, index_col=0)

    ard_df = ard_df.reset_index(drop=True)
    ard_df.index = range(1, len(ard_df) + 1)
    ard_df.index.name = "node_position"

    ard_tensor = torch.tensor(
        ard_df.to_numpy(),
        dtype=torch.float32
    )

    return ard_df, ard_tensor


# ------------------------------------------------------------
# INPUT 1 - All original Facebook profile features
# ------------------------------------------------------------

all_features_df, all_features_tensor = load_features_all(
    features_path
)


# ------------------------------------------------------------
# INPUT 2 - ARD circles-only features
# ------------------------------------------------------------

circles_ard_df, circles_tensor = load_ard_features(
    ard_path
)


# ------------------------------------------------------------
# INPUT 3 - Graph structure only
#
# Identity matrix:
# no profile features and no ARD information.
#
# Each node receives a technical unique identifier so DGI can
# learn solely from graph connectivity.
#
# Do not use an all-ones matrix: DGI corruption shuffles rows,
# so an all-ones input would remain unchanged.
# ------------------------------------------------------------

n_nodes = all_features_tensor.shape[0]

structure_only_df = pd.DataFrame(
    np.eye(n_nodes, dtype=np.float32),
    index=range(1, n_nodes + 1),
    columns=[
        f"node_identity_{node_id}"
        for node_id in range(1, n_nodes + 1)
    ]
)

structure_only_df.index.name = "node_position"

structure_only_tensor = torch.tensor(
    structure_only_df.to_numpy(),
    dtype=torch.float32
)

assert all_features_tensor.shape[0] == circles_tensor.shape[0], (
    "All-features and ARD matrices must have the same number of nodes."
)

assert all_features_tensor.shape[0] == structure_only_tensor.shape[0], (
    "Identity matrix must have the same number of nodes as all features."
)

assert structure_only_tensor.shape == (n_nodes, n_nodes), (
    "Identity matrix must have shape n_nodes × n_nodes."
)

assert (structure_only_df.sum(axis=1) == 1).all(), (
    "Each node must have exactly one active identity feature."
)


# ------------------------------------------------------------
# Build feature set registry
# ------------------------------------------------------------

feature_sets = {
    "all_features": {
        "df": all_features_df,
        "tensor": all_features_tensor,
        "columns": list(all_features_df.columns)
    },

    "circles_only": {
        "df": circles_ard_df,
        "tensor": circles_tensor,
        "columns": list(circles_ard_df.columns)
    },

    "graph_structure_identity_only": {
        "df": structure_only_df,
        "tensor": structure_only_tensor,
        "columns": list(structure_only_df.columns)
    }
}


for name, obj in feature_sets.items():
    print("\nFeature set:", name)
    print("Shape:", obj["tensor"].shape)
    print("First columns:", obj["columns"][:10])


Feature set: all_features
Shape: torch.Size([333, 224])
First columns: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']

Feature set: circles_only
Shape: torch.Size([333, 7])
First columns: ['circle15', 'circle16', 'circle11', 'circle0', 'circle6', 'circle4', 'circleNA']

Feature set: graph_structure_identity_only
Shape: torch.Size([333, 333])
First columns: ['node_identity_1', 'node_identity_2', 'node_identity_3', 'node_identity_4', 'node_identity_5', 'node_identity_6', 'node_identity_7', 'node_identity_8', 'node_identity_9', 'node_identity_10']


In [6]:
# ============================================================
# CELL 6 - Alignment checks
# ============================================================

def check_graph_feature_alignment(G, features_df, features_tensor):
    print("Graph nodes:", G.number_of_nodes())
    print("Feature rows:", features_tensor.shape[0])
    print("First graph nodes:", list(G.nodes())[:10])
    print("First feature index:", list(features_df.index)[:10])

    assert G.number_of_nodes() == features_tensor.shape[0], (
        "Mismatch: graph nodes and feature rows differ."
    )

    assert list(G.nodes()) == list(features_df.index), (
        "Mismatch: graph node order and feature row order differ."
    )

    print("Graph-feature alignment OK.")


def check_all_graphs_alignment(graphs, features_df, features_tensor):
    for i, G in enumerate(graphs, start=1):
        assert G.number_of_nodes() == features_tensor.shape[0], (
            f"Graph {i}: number of nodes differs from features."
        )

        assert list(G.nodes()) == list(features_df.index), (
            f"Graph {i}: node order differs from feature order."
        )

    print(
        f"All {len(graphs)} synthetic graphs are aligned "
        "with the input matrix."
    )


for feature_name, obj in feature_sets.items():
    print(f"\nChecking alignment for: {feature_name}")

    check_graph_feature_alignment(
        G=graphs[0],
        features_df=obj["df"],
        features_tensor=obj["tensor"]
    )

    check_all_graphs_alignment(
        graphs=graphs,
        features_df=obj["df"],
        features_tensor=obj["tensor"]
    )


Checking alignment for: all_features
Graph nodes: 333
Feature rows: 333
First graph nodes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
First feature index: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Graph-feature alignment OK.
All 450 synthetic graphs are aligned with the input matrix.

Checking alignment for: circles_only
Graph nodes: 333
Feature rows: 333
First graph nodes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
First feature index: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Graph-feature alignment OK.
All 450 synthetic graphs are aligned with the input matrix.

Checking alignment for: graph_structure_identity_only
Graph nodes: 333
Feature rows: 333
First graph nodes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
First feature index: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Graph-feature alignment OK.
All 450 synthetic graphs are aligned with the input matrix.


In [7]:
# ============================================================
# CELL 7 - Convert NetworkX graphs to PyG
# ============================================================

def graph_to_pyg_data(G, features_tensor):
    if G.number_of_nodes() != features_tensor.shape[0]:
        raise ValueError(
            f"Mismatch: graph has {G.number_of_nodes()} nodes, "
            f"features have {features_tensor.shape[0]} rows."
        )

    data = from_networkx(G)
    data.x = features_tensor

    return data


def graphs_to_pyg_data_list(graphs, features_tensor):
    data_list = []

    for G in tqdm(graphs, desc="Converting graphs to PyG"):
        data = graph_to_pyg_data(G, features_tensor)
        data_list.append(data)

    return data_list

In [8]:
# ============================================================
# CELL 8 - GCN Encoder and DGI model
# ============================================================

class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = GCNConv(in_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        return self.conv(x, edge_index).relu()


def corruption(x, edge_index, batch=None):
    """
    DGI corruption:
    - keeps graph structure fixed
    - shuffles node features
    - compatible with both single graph and batched graphs
    """
    perm = torch.randperm(x.size(0), device=x.device)
    return x[perm], edge_index, batch


def build_dgi_model(num_features, embedding_dim):
    encoder = GCNEncoder(
        in_channels=num_features,
        out_channels=embedding_dim
    )

    model = DeepGraphInfomax(
        hidden_channels=embedding_dim,
        encoder=encoder,
        summary=lambda z, *args, **kwargs: torch.sigmoid(z.mean(dim=0)),
        corruption=corruption,
    )

    return model

In [9]:
# ============================================================
# CELL 9 - Save utilities
# ============================================================

def save_embeddings(embeddings, output_path):
    np.save(output_path, embeddings)


def save_loss_histories(loss_histories, output_path):
    with open(output_path, "wb") as f:
        pickle.dump(loss_histories, f)


def save_model_state(model, output_path):
    torch.save(model.state_dict(), output_path)

In [10]:
# ============================================================
# CELL 10 - METHOD 1
# Original algorithm: independent DGI per synthetic graph + mean
# ============================================================

def train_dgi_single_graph(
    G,
    features_tensor,
    embedding_dim=64,
    lr=0.001,
    max_epochs=500,
    patience=20,
    seed=123,
    verbose=True,
):
    if seed is not None:
        set_seed(seed)

    device = get_device()

    data = graph_to_pyg_data(G, features_tensor).to(device)

    model = build_dgi_model(
        num_features=data.num_features,
        embedding_dim=embedding_dim
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_loss = float("inf")
    patience_counter = 0
    loss_history = []

    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad()

        pos_z, neg_z, summary = model(data.x, data.edge_index)
        loss = model.loss(pos_z, neg_z, summary)

        loss.backward()
        optimizer.step()

        loss_value = loss.item()
        loss_history.append(loss_value)

        if loss_value < best_loss:
            best_loss = loss_value
            patience_counter = 0
        else:
            patience_counter += 1

        if verbose and epoch % 50 == 0:
            print(f"Epoch {epoch:03d} | Loss: {loss_value:.4f}")

        if patience_counter >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch}")
            break

    model.eval()
    with torch.no_grad():
        embeddings = model.encoder(
            data.x,
            data.edge_index
        ).cpu().numpy()

    return embeddings, loss_history


def train_dgi_ensemble_mean_independent(
    graphs,
    features_tensor,
    embedding_dim=64,
    lr=0.001,
    max_epochs=500,
    patience=20,
    seed=123,
    verbose=False,
):
    """
    Original methodology:
    - one independent DGI model per synthetic graph
    - one latent space per graph
    - average embeddings across graphs

    This reproduces the original notebook logic.
    """
    set_seed(seed)

    if len(graphs) == 0:
        raise ValueError("No graphs provided.")

    n_nodes = features_tensor.shape[0]
    embeddings_sum = np.zeros((n_nodes, embedding_dim), dtype=np.float32)
    all_loss_histories = []

    for i, G in enumerate(tqdm(graphs, desc="Independent DGI ensemble"), start=1):
        if verbose:
            print(f"\n=== Graph {i}/{len(graphs)} ===")

        embeddings, loss_history = train_dgi_single_graph(
            G=G,
            features_tensor=features_tensor,
            embedding_dim=embedding_dim,
            lr=lr,
            max_epochs=max_epochs,
            patience=patience,
            seed=None,
            verbose=verbose,
        )

        embeddings_sum += embeddings
        all_loss_histories.append(loss_history)

    embeddings_mean = embeddings_sum / len(graphs)

    return embeddings_mean, all_loss_histories

In [11]:
# ============================================================
# CELL 11 - METHOD 2
# Shared DGI trained sequentially across all synthetic graphs
#
# NEW:
# We additionally retain the node embeddings produced DURING
# the best training epoch, before re-applying the final encoder.
# ============================================================

def train_dgi_shared_across_graphs(
    graphs,
    features_tensor,
    embedding_dim=64,
    lr=0.001,
    max_epochs=500,
    patience=20,
    seed=123,
    verbose=True,
):
    """
    One Encoder methodology:
    - one single DGI model
    - same encoder for all synthetic graphs
    - training is sequential over graphs each epoch

    Two outputs are produced:

    1) post_training_embeddings_mean:
       after training, the shared encoder is applied again to every
       reconstructed graph and node embeddings are averaged.

    2) training_embeddings_mean:
       embeddings generated directly during the epoch with the
       lowest mean training loss are averaged by node across graphs.
       No additional encoder pass is used for this second output.

    Important:
    during One Encoder training, the encoder parameters change after
    every graph. Therefore, the training-time embeddings of different
    graphs within the same epoch are produced at successive states of
    the shared encoder. This is intentional in this diagnostic experiment.
    """
    set_seed(seed)

    if len(graphs) == 0:
        raise ValueError("No graphs provided.")

    device = get_device()

    data_list = graphs_to_pyg_data_list(
        graphs=graphs,
        features_tensor=features_tensor
    )

    data_list = [data.to(device) for data in data_list]

    num_features = data_list[0].num_features

    model = build_dgi_model(
        num_features=num_features,
        embedding_dim=embedding_dim
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    best_loss = float("inf")
    best_training_embeddings_mean = None
    best_training_epoch = None

    patience_counter = 0
    epoch_loss_history = []

    for epoch in range(max_epochs):
        model.train()

        graph_indices = np.random.permutation(
            len(data_list)
        )

        epoch_losses = []

        # Store the positive embeddings generated during this epoch.
        # Indexing by graph ID restores graph order despite shuffling.
        epoch_training_embeddings = [
            None
        ] * len(data_list)

        for idx in graph_indices:
            data = data_list[idx]

            optimizer.zero_grad()

            pos_z, neg_z, summary = model(
                data.x,
                data.edge_index
            )

            # IMPORTANT:
            # pos_z is the embedding produced DURING training,
            # before the optimizer updates the encoder parameters.
            epoch_training_embeddings[idx] = (
                pos_z.detach().cpu().numpy().copy()
            )

            loss = model.loss(
                pos_z,
                neg_z,
                summary
            )

            loss.backward()
            optimizer.step()

            epoch_losses.append(
                loss.item()
            )

        epoch_mean_loss = float(
            np.mean(epoch_losses)
        )

        epoch_loss_history.append(
            epoch_mean_loss
        )

        if epoch_mean_loss < best_loss:
            best_loss = epoch_mean_loss
            patience_counter = 0
            best_training_epoch = epoch

            if any(
                emb is None
                for emb in epoch_training_embeddings
            ):
                raise RuntimeError(
                    "Missing training embeddings for at least one graph."
                )

            # Same node order in every reconstruction:
            # mean across graph realizations -> one embedding per node.
            best_training_embeddings_mean = np.mean(
                np.stack(
                    epoch_training_embeddings,
                    axis=0
                ),
                axis=0
            ).astype(np.float32)

        else:
            patience_counter += 1

        if verbose and epoch % 10 == 0:
            print(
                f"Epoch {epoch:03d} | "
                f"Mean loss across graphs: "
                f"{epoch_mean_loss:.4f}"
            )

        if patience_counter >= patience:
            if verbose:
                print(
                    f"Early stopping at epoch {epoch}"
                )
            break

    if best_training_embeddings_mean is None:
        raise RuntimeError(
            "No training-time embeddings were retained."
        )

    # --------------------------------------------------------
    # ORIGINAL OUTPUT:
    # Re-apply the trained shared encoder to every graph.
    # --------------------------------------------------------
    model.eval()

    n_nodes = features_tensor.shape[0]

    embeddings_sum = np.zeros(
        (n_nodes, embedding_dim),
        dtype=np.float32
    )

    all_graph_embeddings = []

    with torch.no_grad():
        for data in tqdm(
            data_list,
            desc="Extracting shared-DGI post-training embeddings"
        ):
            emb = model.encoder(
                data.x,
                data.edge_index
            ).cpu().numpy()

            embeddings_sum += emb
            all_graph_embeddings.append(emb)

    post_training_embeddings_mean = (
        embeddings_sum
        / len(data_list)
    )

    return (
        post_training_embeddings_mean,
        all_graph_embeddings,
        epoch_loss_history,
        model,
        best_training_embeddings_mean,
        best_training_epoch,
        best_loss,
    )


In [12]:
# ============================================================
# CELL 12 - METHOD 3
# Batch DGI trained across all synthetic graphs
#
# NEW:
# We additionally retain the node embeddings produced DURING
# the best training epoch, before re-applying the final encoder.
# ============================================================

def _mean_training_embeddings_from_pyg_batch(
    positive_embeddings,
    batch_vector,
    n_nodes
):
    """
    Convert PyG batched node embeddings into one N x d matrix.

    Each synthetic graph contains the same nodes in the same order.
    We split the batched embeddings by graph and average the
    corresponding node embeddings across graph realizations.
    """
    graph_embeddings = []

    graph_ids = torch.unique(
        batch_vector,
        sorted=True
    )

    for graph_id in graph_ids:
        mask = (
            batch_vector == graph_id
        )

        emb_graph = positive_embeddings[
            mask
        ]

        if emb_graph.shape[0] != n_nodes:
            raise ValueError(
                "A graph in the batch does not contain the "
                "expected number of nodes."
            )

        graph_embeddings.append(
            emb_graph.detach().cpu().numpy().copy()
        )

    return graph_embeddings


def train_dgi_batch_across_graphs(
    graphs,
    features_tensor,
    embedding_dim=64,
    lr=0.001,
    max_epochs=500,
    patience=20,
    seed=123,
    batch_size=None,
    verbose=True,
):
    """
    Batch methodology:
    - one single DGI model
    - same encoder for all synthetic graphs
    - graphs are passed using PyG batches
    - by default, batch_size = number of graphs (full batch)

    Two outputs are produced:

    1) post_training_embeddings_mean:
       after training, the shared encoder is applied again to every
       reconstructed graph and node embeddings are averaged.

    2) training_embeddings_mean:
       embeddings produced directly during the epoch with the
       lowest training loss are averaged by node across graphs.
       No additional encoder pass is used for this second output.
    """
    set_seed(seed)

    if len(graphs) == 0:
        raise ValueError(
            "No graphs provided."
        )

    device = get_device()

    data_list = graphs_to_pyg_data_list(
        graphs=graphs,
        features_tensor=features_tensor
    )

    if batch_size is None:
        batch_size = len(data_list)

    loader = DataLoader(
        data_list,
        batch_size=batch_size,
        shuffle=True
    )

    num_features = data_list[0].num_features
    n_nodes = features_tensor.shape[0]

    model = build_dgi_model(
        num_features=num_features,
        embedding_dim=embedding_dim
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    best_loss = float("inf")
    best_training_embeddings_mean = None
    best_training_epoch = None

    patience_counter = 0
    epoch_loss_history = []

    for epoch in range(max_epochs):
        model.train()

        batch_losses = []

        # Collect embeddings from every graph as they are produced
        # during this epoch.
        epoch_training_graph_embeddings = []

        for batch in loader:
            batch = batch.to(device)

            optimizer.zero_grad()

            pos_z, neg_z, summary = model(
                batch.x,
                batch.edge_index,
                batch.batch
            )

            # Save embeddings BEFORE optimizer.step().
            epoch_training_graph_embeddings.extend(
                _mean_training_embeddings_from_pyg_batch(
                    positive_embeddings=pos_z,
                    batch_vector=batch.batch,
                    n_nodes=n_nodes
                )
            )

            loss = model.loss(
                pos_z,
                neg_z,
                summary
            )

            loss.backward()
            optimizer.step()

            batch_losses.append(
                loss.item()
            )

        epoch_mean_loss = float(
            np.mean(batch_losses)
        )

        epoch_loss_history.append(
            epoch_mean_loss
        )

        if epoch_mean_loss < best_loss:
            best_loss = epoch_mean_loss
            patience_counter = 0
            best_training_epoch = epoch

            if (
                len(epoch_training_graph_embeddings)
                != len(data_list)
            ):
                raise RuntimeError(
                    "The training epoch did not produce embeddings "
                    "for every reconstructed graph."
                )

            # Mean by node across all reconstructed graphs.
            best_training_embeddings_mean = np.mean(
                np.stack(
                    epoch_training_graph_embeddings,
                    axis=0
                ),
                axis=0
            ).astype(np.float32)

        else:
            patience_counter += 1

        if verbose and epoch % 10 == 0:
            print(
                f"Epoch {epoch:03d} | "
                f"Mean batch loss: "
                f"{epoch_mean_loss:.4f}"
            )

        if patience_counter >= patience:
            if verbose:
                print(
                    f"Early stopping at epoch {epoch}"
                )
            break

    if best_training_embeddings_mean is None:
        raise RuntimeError(
            "No training-time Batch embeddings were retained."
        )

    # --------------------------------------------------------
    # ORIGINAL OUTPUT:
    # Re-apply the trained encoder separately to every graph.
    # --------------------------------------------------------
    model.eval()

    embeddings_sum = np.zeros(
        (n_nodes, embedding_dim),
        dtype=np.float32
    )

    all_graph_embeddings = []

    with torch.no_grad():
        for data in tqdm(
            data_list,
            desc="Extracting batch-DGI post-training embeddings"
        ):
            data = data.to(device)

            emb = model.encoder(
                data.x,
                data.edge_index
            ).cpu().numpy()

            embeddings_sum += emb
            all_graph_embeddings.append(emb)

    post_training_embeddings_mean = (
        embeddings_sum
        / len(data_list)
    )

    return (
        post_training_embeddings_mean,
        all_graph_embeddings,
        epoch_loss_history,
        model,
        best_training_embeddings_mean,
        best_training_epoch,
        best_loss,
    )


In [13]:
# ============================================================
# CELL 13 - Load and reindex original graph
# ============================================================

def load_original_graph_reindexed(edges_path, features_df):
    edges_df = pd.read_csv(
        edges_path,
        sep=" ",
        header=None,
        names=["from", "to"]
    )

    edges_set = set(
        tuple(sorted(edge))
        for edge in edges_df.to_numpy()
    )

    G_temp = nx.Graph()
    G_temp.add_edges_from(edges_set)

    original_nodes_sorted = sorted(G_temp.nodes())

    mapping = {
        old_node: new_node
        for new_node, old_node in enumerate(original_nodes_sorted, start=1)
    }

    G_temp = nx.relabel_nodes(G_temp, mapping)

    G_original = nx.Graph()

    # Force exact node order used in features_df
    G_original.add_nodes_from(list(features_df.index))
    G_original.add_edges_from(G_temp.edges())

    return G_original


G_original = load_original_graph_reindexed(
    edges_path=original_edges_path,
    features_df=feature_sets["all_features"]["df"]
)

print("Original graph nodes:", G_original.number_of_nodes())
print("Original graph edges:", G_original.number_of_edges())

Original graph nodes: 333
Original graph edges: 2519


In [14]:
# ============================================================
# CELL 14 - Run METHOD 1 for all feature sets
# ============================================================

method1_results = {}

for feature_name, obj in feature_sets.items():
    print("\n" + "=" * 70)
    print(f"METHOD 1 - Independent DGI ensemble | Features: {feature_name}")
    print("=" * 70)

    embeddings_mean, loss_histories = train_dgi_ensemble_mean_independent(
        graphs=graphs,
        features_tensor=obj["tensor"],
        embedding_dim=64,
        lr=0.001,
        max_epochs=500,
        patience=20,
        seed=123,
        verbose=False,
    )

    method1_results[feature_name] = {
        "embeddings": embeddings_mean,
        "loss_histories": loss_histories
    }

    print("Embeddings shape:", embeddings_mean.shape)

    save_embeddings(
        embeddings_mean,
        os.path.join(
            output_folder,
            f"embeddings_method1_independent_mean_{feature_name}.npy"
        )
    )

    save_loss_histories(
        loss_histories,
        os.path.join(
            output_folder,
            f"dgi_loss_method1_independent_{feature_name}.pkl"
        )
    )

    print(f"Saved METHOD 1 outputs for {feature_name}.")


METHOD 1 - Independent DGI ensemble | Features: all_features


Independent DGI ensemble: 100%|██████████| 450/450 [19:54<00:00,  2.65s/it]


Embeddings shape: (333, 64)
Saved METHOD 1 outputs for all_features.

METHOD 1 - Independent DGI ensemble | Features: circles_only


Independent DGI ensemble: 100%|██████████| 450/450 [10:06<00:00,  1.35s/it]


Embeddings shape: (333, 64)
Saved METHOD 1 outputs for circles_only.

METHOD 1 - Independent DGI ensemble | Features: graph_structure_identity_only


Independent DGI ensemble: 100%|██████████| 450/450 [22:21<00:00,  2.98s/it]

Embeddings shape: (333, 64)
Saved METHOD 1 outputs for graph_structure_identity_only.


In [15]:
# ============================================================
# CELL 15 - Run METHOD 2 for all feature sets
# ============================================================

method2_results = {}

for feature_name, obj in feature_sets.items():
    print("\n" + "=" * 70)
    print(
        f"METHOD 2 - Shared sequential DGI | "
        f"Features: {feature_name}"
    )
    print("=" * 70)

    (
        embeddings_mean,
        all_graph_embeddings,
        loss_history,
        model,
        training_embeddings_mean,
        training_best_epoch,
        training_best_loss,
    ) = train_dgi_shared_across_graphs(
        graphs=graphs,
        features_tensor=obj["tensor"],
        embedding_dim=64,
        lr=0.001,
        max_epochs=500,
        patience=20,
        seed=123,
        verbose=True,
    )

    method2_results[feature_name] = {
        "embeddings": embeddings_mean,
        "training_embeddings": training_embeddings_mean,
        "all_graph_embeddings": all_graph_embeddings,
        "loss_history": loss_history,
        "model": model,
        "training_best_epoch": training_best_epoch,
        "training_best_loss": training_best_loss,
    }

    print(
        "Post-training embeddings shape:",
        embeddings_mean.shape
    )
    print(
        "Training-time embeddings shape:",
        training_embeddings_mean.shape
    )
    print(
        "Training-time best epoch:",
        training_best_epoch
    )
    print(
        "Training-time best loss:",
        training_best_loss
    )

    # Original output: re-encoded after training.
    save_embeddings(
        embeddings_mean,
        os.path.join(
            output_folder,
            f"embeddings_method2_shared_sequential_mean_{feature_name}.npy"
        )
    )

    # NEW output: directly from training.
    save_embeddings(
        training_embeddings_mean,
        os.path.join(
            output_folder,
            f"embeddings_method2_shared_sequential_TRAINING_MEAN_{feature_name}.npy"
        )
    )

    save_loss_histories(
        loss_history,
        os.path.join(
            output_folder,
            f"dgi_loss_method2_shared_sequential_{feature_name}.pkl"
        )
    )

    save_model_state(
        model,
        os.path.join(
            output_folder,
            f"dgi_model_method2_shared_sequential_{feature_name}.pt"
        )
    )

    print(
        f"Saved METHOD 2 post-training and "
        f"training-time outputs for {feature_name}."
    )



METHOD 2 - Shared sequential DGI | Features: all_features


Converting graphs to PyG: 100%|██████████| 450/450 [00:45<00:00, 10.00it/s]


Epoch 000 | Mean loss across graphs: 1.0417
Epoch 010 | Mean loss across graphs: 0.6833
Epoch 020 | Mean loss across graphs: 0.6171
Epoch 030 | Mean loss across graphs: 0.5898
Epoch 040 | Mean loss across graphs: 0.5822
Epoch 050 | Mean loss across graphs: 0.5670
Epoch 060 | Mean loss across graphs: 0.5566
Epoch 070 | Mean loss across graphs: 0.5512
Epoch 080 | Mean loss across graphs: 0.5467
Epoch 090 | Mean loss across graphs: 0.5381
Epoch 100 | Mean loss across graphs: 0.5415
Epoch 110 | Mean loss across graphs: 0.5352
Epoch 120 | Mean loss across graphs: 0.5295
Epoch 130 | Mean loss across graphs: 0.5270
Epoch 140 | Mean loss across graphs: 0.5169
Epoch 150 | Mean loss across graphs: 0.5250
Epoch 160 | Mean loss across graphs: 0.5159
Epoch 170 | Mean loss across graphs: 0.5212
Epoch 180 | Mean loss across graphs: 0.5170
Epoch 190 | Mean loss across graphs: 0.5085
Epoch 200 | Mean loss across graphs: 0.5122
Epoch 210 | Mean loss across graphs: 0.5113
Epoch 220 | Mean loss across gra

Extracting shared-DGI post-training embeddings: 100%|██████████| 450/450 [00:00<00:00, 568.32it/s]


Post-training embeddings shape: (333, 64)
Training-time embeddings shape: (333, 64)
Training-time best epoch: 318
Training-time best loss: 0.4901663561662038
Saved METHOD 2 post-training and training-time outputs for all_features.

METHOD 2 - Shared sequential DGI | Features: circles_only


Converting graphs to PyG: 100%|██████████| 450/450 [00:44<00:00, 10.04it/s]


... (3 outputs left out)

Converting graphs to PyG: 100%|██████████| 450/450 [00:43<00:00, 10.29it/s]


Epoch 000 | Mean loss across graphs: 0.9800
Epoch 010 | Mean loss across graphs: 0.4995
Epoch 020 | Mean loss across graphs: 0.4364
Epoch 030 | Mean loss across graphs: 0.4166
Epoch 040 | Mean loss across graphs: 0.4034
Epoch 050 | Mean loss across graphs: 0.3926
Epoch 060 | Mean loss across graphs: 0.3818
Epoch 070 | Mean loss across graphs: 0.3795
Epoch 080 | Mean loss across graphs: 0.3755
Epoch 090 | Mean loss across graphs: 0.3700
Epoch 100 | Mean loss across graphs: 0.3734
Epoch 110 | Mean loss across graphs: 0.3699
Epoch 120 | Mean loss across graphs: 0.3632
Epoch 130 | Mean loss across graphs: 0.3655
Epoch 140 | Mean loss across graphs: 0.3570
Epoch 150 | Mean loss across graphs: 0.3562
Epoch 160 | Mean loss across graphs: 0.3587
Epoch 170 | Mean loss across graphs: 0.3526
Epoch 180 | Mean loss across graphs: 0.3551
Epoch 190 | Mean loss across graphs: 0.3535
Early stopping at epoch 192


Extracting shared-DGI post-training embeddings: 100%|██████████| 450/450 [00:00<00:00, 482.87it/s]

Post-training embeddings shape: (333, 64)
Training-time embeddings shape: (333, 64)
Training-time best epoch: 172
Training-time best loss: 0.35117992993858127
Saved METHOD 2 post-training and training-time outputs for graph_structure_identity_only.


In [16]:
# ============================================================
# CELL 16 - Run METHOD 3 for all feature sets
# ============================================================

method3_results = {}

for feature_name, obj in feature_sets.items():
    print("\n" + "=" * 70)
    print(
        f"METHOD 3 - Batch DGI | "
        f"Features: {feature_name}"
    )
    print("=" * 70)

    (
        embeddings_mean,
        all_graph_embeddings,
        loss_history,
        model,
        training_embeddings_mean,
        training_best_epoch,
        training_best_loss,
    ) = train_dgi_batch_across_graphs(
        graphs=graphs,
        features_tensor=obj["tensor"],
        embedding_dim=64,
        lr=0.001,
        max_epochs=500,
        patience=20,
        seed=123,
        batch_size=None,   # None = all graphs in one batch
        verbose=True,
    )

    method3_results[feature_name] = {
        "embeddings": embeddings_mean,
        "training_embeddings": training_embeddings_mean,
        "all_graph_embeddings": all_graph_embeddings,
        "loss_history": loss_history,
        "model": model,
        "training_best_epoch": training_best_epoch,
        "training_best_loss": training_best_loss,
    }

    print(
        "Post-training embeddings shape:",
        embeddings_mean.shape
    )
    print(
        "Training-time embeddings shape:",
        training_embeddings_mean.shape
    )
    print(
        "Training-time best epoch:",
        training_best_epoch
    )
    print(
        "Training-time best loss:",
        training_best_loss
    )

    # Original output: re-encoded after training.
    save_embeddings(
        embeddings_mean,
        os.path.join(
            output_folder,
            f"embeddings_method3_batch_mean_{feature_name}.npy"
        )
    )

    # NEW output: directly from training.
    save_embeddings(
        training_embeddings_mean,
        os.path.join(
            output_folder,
            f"embeddings_method3_batch_TRAINING_MEAN_{feature_name}.npy"
        )
    )

    save_loss_histories(
        loss_history,
        os.path.join(
            output_folder,
            f"dgi_loss_method3_batch_{feature_name}.pkl"
        )
    )

    save_model_state(
        model,
        os.path.join(
            output_folder,
            f"dgi_model_method3_batch_{feature_name}.pt"
        )
    )

    print(
        f"Saved METHOD 3 post-training and "
        f"training-time outputs for {feature_name}."
    )



METHOD 3 - Batch DGI | Features: all_features


Converting graphs to PyG: 100%|██████████| 450/450 [00:44<00:00, 10.04it/s]


Epoch 000 | Mean batch loss: 1.3987
Epoch 010 | Mean batch loss: 1.3589
Epoch 020 | Mean batch loss: 1.3196
Epoch 030 | Mean batch loss: 1.2635
Epoch 040 | Mean batch loss: 1.1992
Epoch 050 | Mean batch loss: 1.1367
Epoch 060 | Mean batch loss: 1.0847
Epoch 070 | Mean batch loss: 1.0410
Epoch 080 | Mean batch loss: 1.0073
Epoch 090 | Mean batch loss: 0.9737
Epoch 100 | Mean batch loss: 0.9509
Epoch 110 | Mean batch loss: 0.9294
Epoch 120 | Mean batch loss: 0.9128
Epoch 130 | Mean batch loss: 0.9000
Epoch 140 | Mean batch loss: 0.8867
Epoch 150 | Mean batch loss: 0.8773
Epoch 160 | Mean batch loss: 0.8642
Epoch 170 | Mean batch loss: 0.8568
Epoch 180 | Mean batch loss: 0.8497
Epoch 190 | Mean batch loss: 0.8411
Epoch 200 | Mean batch loss: 0.8298
Epoch 210 | Mean batch loss: 0.8235
Epoch 220 | Mean batch loss: 0.8142
Epoch 230 | Mean batch loss: 0.8118
Epoch 240 | Mean batch loss: 0.8060
Epoch 250 | Mean batch loss: 0.8013
Epoch 260 | Mean batch loss: 0.7956
Epoch 270 | Mean batch loss:

Extracting batch-DGI post-training embeddings: 100%|██████████| 450/450 [00:00<00:00, 510.54it/s]


Post-training embeddings shape: (333, 64)
Training-time embeddings shape: (333, 64)
Training-time best epoch: 497
Training-time best loss: 0.7053648233413696
Saved METHOD 3 post-training and training-time outputs for all_features.

METHOD 3 - Batch DGI | Features: circles_only


Converting graphs to PyG: 100%|██████████| 450/450 [00:48<00:00,  9.36it/s]


... (3 outputs left out)

Converting graphs to PyG: 100%|██████████| 450/450 [00:45<00:00,  9.91it/s]


Epoch 000 | Mean batch loss: 1.3869
Epoch 010 | Mean batch loss: 1.3679
Epoch 020 | Mean batch loss: 1.3253
Epoch 030 | Mean batch loss: 1.2695
Epoch 040 | Mean batch loss: 1.2132
Epoch 050 | Mean batch loss: 1.1564
Epoch 060 | Mean batch loss: 1.0994
Epoch 070 | Mean batch loss: 1.0467
Epoch 080 | Mean batch loss: 0.9970
Epoch 090 | Mean batch loss: 0.9548
Epoch 100 | Mean batch loss: 0.9164
Epoch 110 | Mean batch loss: 0.8878
Epoch 120 | Mean batch loss: 0.8636
Epoch 130 | Mean batch loss: 0.8417
Epoch 140 | Mean batch loss: 0.8196
Epoch 150 | Mean batch loss: 0.8011
Epoch 160 | Mean batch loss: 0.7876
Epoch 170 | Mean batch loss: 0.7752
Epoch 180 | Mean batch loss: 0.7607
Epoch 190 | Mean batch loss: 0.7493
Epoch 200 | Mean batch loss: 0.7381
Epoch 210 | Mean batch loss: 0.7231
Epoch 220 | Mean batch loss: 0.7137
Epoch 230 | Mean batch loss: 0.7018
Epoch 240 | Mean batch loss: 0.6902
Epoch 250 | Mean batch loss: 0.6856
Epoch 260 | Mean batch loss: 0.6764
Epoch 270 | Mean batch loss:

Extracting batch-DGI post-training embeddings: 100%|██████████| 450/450 [00:00<00:00, 461.92it/s]

Post-training embeddings shape: (333, 64)
Training-time embeddings shape: (333, 64)
Training-time best epoch: 498
Training-time best loss: 0.5450282096862793
Saved METHOD 3 post-training and training-time outputs for graph_structure_identity_only.


In [17]:
# ============================================================
# Helper - Original graph: retain training-time embeddings
# ============================================================

def train_dgi_single_graph_with_training_embeddings(
    G,
    features_tensor,
    embedding_dim=64,
    lr=0.001,
    max_epochs=500,
    patience=20,
    seed=123,
    verbose=True,
):
    """
    Train DGI on a single graph and return:
      1) the current post-training embeddings,
      2) the loss history,
      3) the positive node embeddings produced directly during
         the best training epoch,
      4) the corresponding epoch,
      5) the corresponding loss.

    The training-time embeddings are captured before optimizer.step().
    """
    if seed is not None:
        set_seed(seed)

    device = get_device()
    data = graph_to_pyg_data(G, features_tensor).to(device)

    model = build_dgi_model(
        num_features=data.num_features,
        embedding_dim=embedding_dim
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_loss = float("inf")
    best_training_embeddings = None
    best_training_epoch = None
    patience_counter = 0
    loss_history = []

    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad()

        pos_z, neg_z, summary = model(data.x, data.edge_index)
        loss = model.loss(pos_z, neg_z, summary)

        loss_value = loss.item()
        loss_history.append(loss_value)

        # Retain the embeddings generated directly during training,
        # before updating the model parameters.
        if loss_value < best_loss:
            best_loss = loss_value
            best_training_epoch = epoch
            best_training_embeddings = (
                pos_z.detach().cpu().numpy().copy()
            )
            patience_counter = 0
        else:
            patience_counter += 1

        loss.backward()
        optimizer.step()

        if verbose and epoch % 50 == 0:
            print(f"Epoch {epoch:03d} | Loss: {loss_value:.4f}")

        if patience_counter >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch}")
            break

    model.eval()
    with torch.no_grad():
        embeddings_post_training = model.encoder(
            data.x,
            data.edge_index
        ).cpu().numpy()

    if best_training_embeddings is None:
        raise RuntimeError(
            "No training-time embeddings were retained for the original graph."
        )

    return (
        embeddings_post_training,
        loss_history,
        best_training_embeddings,
        best_training_epoch,
        best_loss,
    )


In [18]:
# ============================================================
# CELL 17 - Train DGI on original graph for all feature sets
# Current post-training embeddings + new training-time embeddings
# ============================================================

original_results = {}

for feature_name, obj in feature_sets.items():
    print("\n" + "=" * 70)
    print(f"ORIGINAL GRAPH DGI | Features: {feature_name}")
    print("=" * 70)

    check_graph_feature_alignment(
        G=G_original,
        features_df=obj["df"],
        features_tensor=obj["tensor"]
    )

    (
        embeddings_original,
        loss_history_original,
        training_embeddings_original,
        training_best_epoch_original,
        training_best_loss_original,
    ) = train_dgi_single_graph_with_training_embeddings(
        G=G_original,
        features_tensor=obj["tensor"],
        embedding_dim=64,
        lr=0.001,
        max_epochs=500,
        patience=20,
        seed=123,
        verbose=True,
    )

    original_results[feature_name] = {
        "embeddings": embeddings_original,
        "training_embeddings": training_embeddings_original,
        "loss_history": loss_history_original,
        "training_best_epoch": training_best_epoch_original,
        "training_best_loss": training_best_loss_original,
    }

    print("Original post-training embeddings shape:", embeddings_original.shape)
    print(
        "Original training-time embeddings shape:",
        training_embeddings_original.shape
    )
    print(
        "Original training-time epoch/loss:",
        training_best_epoch_original,
        training_best_loss_original
    )

    # Existing post-training embeddings
    save_embeddings(
        embeddings_original,
        os.path.join(
            output_folder,
            f"embeddings_original_{feature_name}.npy"
        )
    )

    # New training-time embeddings.
    # No averaging is required because there is only one original graph.
    save_embeddings(
        training_embeddings_original,
        os.path.join(
            output_folder,
            f"embeddings_original_TRAINING_{feature_name}.npy"
        )
    )

    save_loss_histories(
        loss_history_original,
        os.path.join(
            output_folder,
            f"dgi_loss_original_{feature_name}.pkl"
        )
    )

    print(f"Saved original graph DGI outputs for {feature_name}.")



ORIGINAL GRAPH DGI | Features: all_features
Graph nodes: 333
Feature rows: 333
First graph nodes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
First feature index: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Graph-feature alignment OK.
Epoch 000 | Loss: 1.3968
Epoch 050 | Loss: 1.1016
Epoch 100 | Loss: 0.5718
Epoch 150 | Loss: 0.2595
Epoch 200 | Loss: 0.1840
Epoch 250 | Loss: 0.1144
Epoch 300 | Loss: 0.0755
Epoch 350 | Loss: 0.0701
Early stopping at epoch 357
Original post-training embeddings shape: (333, 64)
Original training-time embeddings shape: (333, 64)
Original training-time epoch/loss: 337 0.054997462779283524
Saved original graph DGI outputs for all_features.

ORIGINAL GRAPH DGI | Features: circles_only
Graph nodes: 333
Feature rows: 333
First graph nodes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
First feature index: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Graph-feature alignment OK.
Epoch 000 | Loss: 3.7175
Epoch 050 | Loss: 1.3916
Early stopping at epoch 87
Original post-training embeddings shape: (333, 64)
Origina

In [19]:
# ============================================================
# CELL 18 - Summary of saved embeddings
# ============================================================

print("Saved embeddings in:", output_folder)

for feature_name in feature_sets.keys():
    print("\nFeature set:", feature_name)

    print(
        "METHOD 1 - current independent mean:",
        os.path.join(
            output_folder,
            f"embeddings_method1_independent_mean_{feature_name}.npy"
        )
    )

    print(
        "METHOD 2 - post-training mean:",
        os.path.join(
            output_folder,
            f"embeddings_method2_shared_sequential_mean_{feature_name}.npy"
        )
    )

    print(
        "METHOD 2 - TRAINING-TIME mean:",
        os.path.join(
            output_folder,
            f"embeddings_method2_shared_sequential_TRAINING_MEAN_{feature_name}.npy"
        )
    )

    print(
        "METHOD 3 - post-training mean:",
        os.path.join(
            output_folder,
            f"embeddings_method3_batch_mean_{feature_name}.npy"
        )
    )

    print(
        "METHOD 3 - TRAINING-TIME mean:",
        os.path.join(
            output_folder,
            f"embeddings_method3_batch_TRAINING_MEAN_{feature_name}.npy"
        )
    )

    print(
        "ORIGINAL:",
        os.path.join(
            output_folder,
            f"embeddings_original_{feature_name}.npy"
        )
    )

    print(
        "ORIGINAL - TRAINING-TIME:",
        os.path.join(
            output_folder,
            f"embeddings_original_TRAINING_{feature_name}.npy"
        )
    )

print(
    "\nNOTE: METHOD 1 is intentionally unchanged. "
    "Each graph has its own independently trained encoder, "
    "and its embedding is already extracted on that same graph "
    "before averaging across reconstructions."
)


Saved embeddings in: facebook0_dgi

Feature set: all_features
METHOD 1 - current independent mean: facebook0_dgi\embeddings_method1_independent_mean_all_features.npy
METHOD 2 - post-training mean: facebook0_dgi\embeddings_method2_shared_sequential_mean_all_features.npy
METHOD 2 - TRAINING-TIME mean: facebook0_dgi\embeddings_method2_shared_sequential_TRAINING_MEAN_all_features.npy
METHOD 3 - post-training mean: facebook0_dgi\embeddings_method3_batch_mean_all_features.npy
METHOD 3 - TRAINING-TIME mean: facebook0_dgi\embeddings_method3_batch_TRAINING_MEAN_all_features.npy
ORIGINAL: facebook0_dgi\embeddings_original_all_features.npy
ORIGINAL - TRAINING-TIME: facebook0_dgi\embeddings_original_TRAINING_all_features.npy

Feature set: circles_only
METHOD 1 - current independent mean: facebook0_dgi\embeddings_method1_independent_mean_circles_only.npy
METHOD 2 - post-training mean: facebook0_dgi\embeddings_method2_shared_sequential_mean_circles_only.npy
METHOD 2 - TRAINING-TIME mean: facebook0_d